# Semana 8 — Pipeline reproducible de punta a punta: paquete, pruebas, linaje y publicación

**Objetivo:** convertir el trabajo de 7 notebooks en un **paquete de Python** (`healthetl`) que ejecuta todo el ETL con un solo comando, con SQL en archivos versionables, pruebas automáticas con `pytest`, registro (`logging`), un **manifiesto de ejecución** con huellas de entradas y salidas, e integración continua en GitHub Actions.

**Productos reales usados:** `pytest`, `logging`, GitHub Actions, DuckDB.

**Al terminar tendrás** dentro de tu carpeta del proyecto: `src/healthetl/`, `tests/`, `pyproject.toml`, `requirements.txt`, `.gitignore`, `.github/workflows/tests.yml` y `data/runs/<run_id>.json`.

**Requisito:** semanas 1 a 7 (para comparar resultados).

### Celda de configuración del entorno (se repite al inicio de cada semana)

**Explicación línea por línea**

- `import os`, `import sys`, `from pathlib import Path`: cargan tres módulos de la librería estándar. `os` lee variables de entorno, `sys` informa la versión de Python y `Path` representa rutas de archivos como objetos. **Por qué así:** `Path` evita concatenar rutas con texto (`"a" + "/" + "b"`), que falla entre sistemas operativos. **En la práctica real** todo código de datos moderno usa `pathlib`.
- `try: import google.colab` … `except ImportError`: intenta importar un módulo que solo existe en Colab. Si la importación funciona, `IN_COLAB = True`; si lanza `ImportError`, estamos en otra máquina. **Por qué importa:** el mismo notebook corre en Colab y en tu laptop sin editarlo. **Cuándo se usa:** siempre que un código deba ser portable entre entornos.
- `from google.colab import drive` y `drive.mount("/content/drive")`: conectan tu Google Drive al sistema de archivos de Colab. **Por qué:** el disco de Colab se borra al cerrar la sesión; en Drive los datos de la semana 1 siguen ahí el lunes siguiente. La primera vez Colab te pedirá autorizar el acceso.
- `PROJECT_DIR = Path("/content/drive/MyDrive/health-data-etl-lab")`: carpeta raíz del proyecto dentro de tu Drive.
- `Path(os.environ.get("HEALTHETL_PROJECT_DIR", ".")).resolve()`: fuera de Colab usa la variable de entorno `HEALTHETL_PROJECT_DIR` si existe; si no, la carpeta actual (`"."`). `.resolve()` la convierte en ruta absoluta. **En la práctica real** la configuración se pasa por variables de entorno, nunca escrita a mano dentro del código (principio *12-factor app*).
- `DATA_DIR = PROJECT_DIR / "data"`: el operador `/` de `pathlib` une rutas de forma segura.
- `DATA_DIR.mkdir(parents=True, exist_ok=True)`: crea la carpeta y sus padres; `exist_ok=True` evita un error si ya existe. Esto hace la celda **idempotente**: puedes ejecutarla 1 o 10 veces con el mismo resultado, una propiedad central de todo pipeline ETL.
- Los tres `print`: confirman versión de Python, entorno y ruta. Registrar el entorno al inicio es un hábito de reproducibilidad.

In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab  # este módulo solo existe dentro de Google Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/health-data-etl-lab")
else:
    PROJECT_DIR = Path(os.environ.get("HEALTHETL_PROJECT_DIR", ".")).resolve()

DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0])
print("¿Colab?:", IN_COLAB)
print("Carpeta de datos:", DATA_DIR)

### Paso 1 — Estructura del repositorio

**Explicación línea por línea**

- `for carpeta in [...]: (PROJECT_DIR / carpeta).mkdir(parents=True, exist_ok=True)`: crea la estructura estándar *src layout*: el código importable vive en `src/healthetl/`, separado de notebooks, pruebas y datos. **Por qué *src layout*:** obliga a instalar el paquete para usarlo, así las pruebas prueban lo mismo que usará cualquier otra persona.
- `%cd {PROJECT_DIR}`: comando mágico que cambia el directorio de trabajo del notebook; los `%%writefile` siguientes escriben rutas relativas a la raíz del proyecto.

In [ ]:
for carpeta in ["src/healthetl/sql", "tests", "notebooks", ".github/workflows", "data/runs"]:
    (PROJECT_DIR / carpeta).mkdir(parents=True, exist_ok=True)
%cd {PROJECT_DIR}

### Paso 2 — Archivos de configuración del proyecto

**Explicación línea por línea**

- `%%writefile .gitignore`: la primera línea de la celda indica a Jupyter que escriba el resto de la celda en ese archivo.
- `data/`: **nunca** se versionan datos, ni sintéticos. `__pycache__/`, `*.pyc`: archivos compilados de Python. `.ipynb_checkpoints/`: copias automáticas de Jupyter. `.env`: archivo típico de secretos locales.
- `%%writefile requirements.txt`: dependencias con versión mínima. `>=` acepta versiones nuevas; en un despliegue de producción se fijan versiones exactas (`==`) con un archivo de bloqueo.
- `%%writefile pyproject.toml`: el estándar moderno para declarar un paquete Python (PEP 621).
    - `[build-system]`: herramienta que construye el paquete (`setuptools`).
    - `[project]`: nombre, versión, versión mínima de Python y dependencias.
    - `[tool.setuptools.packages.find] where = ["src"]`: busca paquetes dentro de `src/`.
    - `[tool.setuptools.package-data] healthetl = ["sql/*.sql"]`: incluye los archivos SQL dentro del paquete instalado.
    - `[tool.pytest.ini_options] testpaths = ["tests"]`: dónde busca pruebas `pytest`.

In [ ]:
%%writefile .gitignore
data/
__pycache__/
*.pyc
.ipynb_checkpoints/
.env

In [ ]:
%%writefile requirements.txt
pandas>=2.2
pyarrow>=15
duckdb>=1.1
pandera>=0.24
requests>=2.31
scikit-learn>=1.4
joblib>=1.3
pytest>=8

In [ ]:
%%writefile pyproject.toml
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "healthetl"
version = "0.1.0"
description = "Pipeline ETL educativo: Synthea -> bronze -> silver -> OMOP CDM v5.4"
requires-python = ">=3.10"
dependencies = ["pandas>=2.2", "pyarrow>=15", "duckdb>=1.1", "requests>=2.31"]

[tool.setuptools.packages.find]
where = ["src"]

[tool.setuptools.package-data]
healthetl = ["sql/*.sql"]

[tool.pytest.ini_options]
testpaths = ["tests"]

### Paso 3 — Módulo `terminology.py`: funciones puras (semana 5)

Una **función pura** no lee archivos ni la red: dada la misma entrada devuelve siempre la misma salida. Son las más fáciles de probar y por eso conviene extraerlas primero.

**Explicación línea por línea**

- `"""..."""` al inicio del archivo: *docstring* del módulo, documentación que muestran `help()` y los editores.
- `_D` y `_F`: tablas de Verhoeff; el guion bajo inicial indica "uso interno del módulo". `_F` se deriva de su primera fila como en la semana 5.
- `def verhoeff_ok(code: str) -> bool:` las anotaciones `: str` y `-> bool` documentan tipos; no cambian la ejecución pero las herramientas las verifican.
- `def loinc_check_digit(body: str) -> int:` y `def loinc_ok(code: str) -> bool:` son el mismo algoritmo Mod 10 de la semana 5.
- `__all__ = [...]`: lista pública del módulo.

In [ ]:
%%writefile src/healthetl/terminology.py
"""Validación sin conexión de identificadores SNOMED CT (Verhoeff) y LOINC (Mod 10)."""
import re

_D = [
    [0, 1, 2, 3, 4, 5, 6, 7, 8, 9], [1, 2, 3, 4, 0, 6, 7, 8, 9, 5],
    [2, 3, 4, 0, 1, 7, 8, 9, 5, 6], [3, 4, 0, 1, 2, 8, 9, 5, 6, 7],
    [4, 0, 1, 2, 3, 9, 5, 6, 7, 8], [5, 9, 8, 7, 6, 0, 4, 3, 2, 1],
    [6, 5, 9, 8, 7, 1, 0, 4, 3, 2], [7, 6, 5, 9, 8, 2, 1, 0, 4, 3],
    [8, 7, 6, 5, 9, 3, 2, 1, 0, 4], [9, 8, 7, 6, 5, 4, 3, 2, 1, 0],
]
_F = [list(range(10)), [1, 5, 7, 6, 2, 8, 3, 0, 9, 4]]
for _i in range(2, 8):
    _F.append([_F[_i - 1][_F[1][_j]] for _j in range(10)])


def verhoeff_ok(code: str) -> bool:
    """True si `code` es un SCTID con dígito verificador Verhoeff válido."""
    if not (isinstance(code, str) and code.isdigit() and 6 <= len(code) <= 18):
        return False
    c = 0
    for i, ch in enumerate(reversed(code)):
        c = _D[c][_F[i % 8][int(ch)]]
    return c == 0


def loinc_check_digit(body: str) -> int:
    """Dígito verificador Mod 10 de la parte numérica de un código LOINC."""
    total = 0
    for i, ch in enumerate(reversed(body)):
        d = int(ch)
        if i % 2 == 0:
            d *= 2
            d = d // 10 + d % 10
        total += d
    return (10 - total % 10) % 10


def loinc_ok(code: str) -> bool:
    """True si `code` tiene forma NNNN-N y su dígito verificador es correcto."""
    if not (isinstance(code, str) and re.fullmatch(r"\d+-\d", code)):
        return False
    body, check = code.split("-")
    return loinc_check_digit(body) == int(check)


__all__ = ["verhoeff_ok", "loinc_check_digit", "loinc_ok"]

### Paso 4 — Módulo `extract.py`: descarga y capa bronze (semana 1)

**Explicación línea por línea**

- `import logging` y `log = logging.getLogger(__name__)`: cada módulo tiene su *logger* con el nombre del módulo (`healthetl.extract`). **Por qué `logging` y no `print`:** los mensajes llevan hora y nivel (`INFO`, `WARNING`, `ERROR`), se pueden enviar a archivo y silenciar por nivel. Es lo que exige un pipeline que corre sin nadie mirando.
- `DATASET_URL`, `TABLES`: constantes del dataset y del alcance (las mismas de la semana 1).
- `def sha256_of(path)`: huella por bloques.
- `def download(url, dest)`: descarga solo si falta (idempotente) y registra con `log.info`.
- `def extract_to_bronze(zip_path, bronze_dir, ingested_at)`: lee cada CSV **directamente desde el ZIP** con `zf.open(...)` sin descomprimir a disco, escribe Parquet con columnas de linaje y devuelve el manifiesto.
- `pd.read_csv(f, dtype=str, keep_default_na=False)`: lectura cruda como en bronze.
- `if Path(m.filename).stem not in TABLES: continue`: alcance y protección *zip slip* (solo se usa el nombre final).

In [ ]:
%%writefile src/healthetl/extract.py
"""Extract & Load: descarga idempotente y escritura de la capa bronze."""
import hashlib
import logging
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd

log = logging.getLogger(__name__)

DATASET_URL = (
    "https://raw.githubusercontent.com/synthetichealth/synthea-sample-data/"
    "main/downloads/synthea_sample_data_csv_nov2021.zip"
)
TABLES = ["patients", "encounters", "conditions", "observations", "medications",
          "procedures", "immunizations", "allergies", "careplans", "organizations", "providers"]


def sha256_of(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def download(url: str, dest: Path) -> Path:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        log.info("Ya existe %s; no se descarga", dest.name)
    else:
        log.info("Descargando %s", url)
        urllib.request.urlretrieve(url, dest)
    return dest


def extract_to_bronze(zip_path: Path, bronze_dir: Path, ingested_at: str) -> list:
    bronze_dir.mkdir(parents=True, exist_ok=True)
    manifest = []
    with zipfile.ZipFile(zip_path) as zf:
        for m in zf.infolist():
            tabla = Path(m.filename).stem
            if not m.filename.endswith(".csv") or tabla not in TABLES:
                continue
            with zf.open(m) as f:
                df = pd.read_csv(f, dtype=str, keep_default_na=False)
            df["_source_file"] = Path(m.filename).name
            df["_ingested_at"] = ingested_at
            dest = bronze_dir / f"{tabla}.parquet"
            df.to_parquet(dest, index=False)
            manifest.append({"table": tabla, "rows": len(df), "path": dest.name})
            log.info("bronze.%s: %s filas", tabla, f"{len(df):,}")
    return manifest

### Paso 5 — SQL en archivos: `silver.sql` y `omop.sql` (semanas 3 y 6)

En la industria el SQL vive en archivos `.sql` versionados, no en cadenas dentro de notebooks: se revisa en *pull requests*, se formatea y se reutiliza. Las sentencias son **las mismas** que explicaste línea por línea en las semanas 3 y 6; aquí solo cambian dos cosas:

- `$bronze_dir`: marcador que Python sustituye por la ruta real con `string.Template`. Se usa `$` en vez de `{}` porque las llaves aparecen a veces en SQL.
- En `omop.sql` se omite la carga opcional de vocabularios Athena para que el pipeline no dependa de archivos con licencia; `*_concept_id` queda en 0 donde no hay regla fija.

In [ ]:
%%writefile src/healthetl/sql/silver.sql
-- Capa silver (semana 3). $bronze_dir lo sustituye Python.
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE OR REPLACE VIEW bronze.patients     AS SELECT * FROM read_parquet('$bronze_dir/patients.parquet');
CREATE OR REPLACE VIEW bronze.encounters   AS SELECT * FROM read_parquet('$bronze_dir/encounters.parquet');
CREATE OR REPLACE VIEW bronze.conditions   AS SELECT * FROM read_parquet('$bronze_dir/conditions.parquet');
CREATE OR REPLACE VIEW bronze.observations AS SELECT * FROM read_parquet('$bronze_dir/observations.parquet');

CREATE OR REPLACE TABLE silver.patients AS
SELECT Id AS patient_id, TRY_CAST(BIRTHDATE AS DATE) AS birth_date,
       TRY_CAST(NULLIF(DEATHDATE, '') AS DATE) AS death_date, GENDER AS sex,
       NULLIF(RACE, '') AS race, NULLIF(ETHNICITY, '') AS ethnicity, NULLIF(MARITAL, '') AS marital_status,
       NULLIF(STATE, '') AS state, NULLIF(COUNTY, '') AS county, _ingested_at
FROM bronze.patients
QUALIFY ROW_NUMBER() OVER (PARTITION BY Id ORDER BY _ingested_at DESC) = 1;

CREATE OR REPLACE TABLE silver.encounters AS
SELECT Id AS encounter_id, PATIENT AS patient_id, ENCOUNTERCLASS AS encounter_class,
       CODE AS encounter_snomed_code, DESCRIPTION AS encounter_description,
       TRY_CAST(START AS TIMESTAMPTZ) AT TIME ZONE 'UTC' AS start_ts,
       TRY_CAST(STOP AS TIMESTAMPTZ) AT TIME ZONE 'UTC' AS stop_ts,
       NULLIF(REASONCODE, '') AS reason_snomed_code, ORGANIZATION AS organization_id, PROVIDER AS provider_id
FROM bronze.encounters
QUALIFY ROW_NUMBER() OVER (PARTITION BY Id ORDER BY START) = 1;

CREATE OR REPLACE TABLE silver.conditions AS
SELECT DISTINCT PATIENT AS patient_id, ENCOUNTER AS encounter_id, CODE AS snomed_code, DESCRIPTION AS description,
       TRY_CAST(START AS DATE) AS start_date, TRY_CAST(NULLIF(STOP, '') AS DATE) AS stop_date
FROM bronze.conditions;

CREATE OR REPLACE TABLE silver.observations AS
SELECT DISTINCT TRY_CAST(DATE AS TIMESTAMPTZ) AT TIME ZONE 'UTC' AS obs_ts, PATIENT AS patient_id,
       NULLIF(ENCOUNTER, '') AS encounter_id, NULLIF(CATEGORY, '') AS category, CODE AS loinc_code,
       DESCRIPTION AS description,
       CASE WHEN TYPE = 'numeric' THEN TRY_CAST(VALUE AS DOUBLE) END AS value_num,
       CASE WHEN TYPE = 'text' THEN VALUE END AS value_text,
       NULLIF(UNITS, '') AS unit, TYPE AS value_type
FROM bronze.observations;

In [ ]:
%%writefile src/healthetl/sql/omop.sql
-- OMOP CDM v5.4 (semana 6), sin vocabularios: *_concept_id = 0 donde no hay regla fija.
CREATE SCHEMA IF NOT EXISTS stg;
CREATE SCHEMA IF NOT EXISTS cdm;
CREATE OR REPLACE TABLE stg.person_map AS
SELECT ROW_NUMBER() OVER (ORDER BY patient_id) AS person_id, patient_id FROM silver.patients;
CREATE OR REPLACE TABLE stg.visit_map AS
SELECT ROW_NUMBER() OVER (ORDER BY encounter_id) AS visit_occurrence_id, encounter_id FROM silver.encounters;

CREATE OR REPLACE TABLE cdm.person AS
SELECT m.person_id,
       CASE p.sex WHEN 'M' THEN 8507 WHEN 'F' THEN 8532 ELSE 0 END AS gender_concept_id,
       EXTRACT(year FROM p.birth_date)::INTEGER AS year_of_birth,
       EXTRACT(month FROM p.birth_date)::INTEGER AS month_of_birth,
       EXTRACT(day FROM p.birth_date)::INTEGER AS day_of_birth,
       CAST(p.birth_date AS TIMESTAMP) AS birth_datetime,
       CASE lower(p.race) WHEN 'white' THEN 8527 WHEN 'black' THEN 8516 WHEN 'asian' THEN 8515 ELSE 0 END AS race_concept_id,
       CASE lower(p.ethnicity) WHEN 'hispanic' THEN 38003563 WHEN 'nonhispanic' THEN 38003564 ELSE 0 END AS ethnicity_concept_id,
       p.patient_id AS person_source_value, p.sex AS gender_source_value,
       p.race AS race_source_value, p.ethnicity AS ethnicity_source_value
FROM silver.patients p JOIN stg.person_map m USING (patient_id);

CREATE OR REPLACE TABLE cdm.observation_period AS
SELECT ROW_NUMBER() OVER (ORDER BY m.person_id) AS observation_period_id, m.person_id,
       MIN(e.start_ts)::DATE AS observation_period_start_date, MAX(e.stop_ts)::DATE AS observation_period_end_date,
       32882 AS period_type_concept_id
FROM silver.encounters e JOIN stg.person_map m USING (patient_id)
GROUP BY m.person_id;

CREATE OR REPLACE TABLE cdm.death AS
SELECT m.person_id, p.death_date, 32817 AS death_type_concept_id
FROM silver.patients p JOIN stg.person_map m USING (patient_id)
WHERE p.death_date IS NOT NULL;

CREATE OR REPLACE TABLE cdm.visit_occurrence AS
SELECT v.visit_occurrence_id, m.person_id,
       CASE WHEN e.encounter_class = 'inpatient' THEN 9201
            WHEN e.encounter_class IN ('emergency', 'urgentcare') THEN 9203
            WHEN e.encounter_class IN ('ambulatory', 'wellness', 'outpatient') THEN 9202
            ELSE 0 END AS visit_concept_id,
       e.start_ts::DATE AS visit_start_date, e.start_ts AS visit_start_datetime,
       e.stop_ts::DATE AS visit_end_date, e.stop_ts AS visit_end_datetime,
       32827 AS visit_type_concept_id, e.encounter_id AS visit_source_value,
       e.encounter_class AS visit_source_concept_name,
       LAG(v.visit_occurrence_id) OVER (PARTITION BY m.person_id ORDER BY e.start_ts) AS preceding_visit_occurrence_id
FROM silver.encounters e
JOIN stg.person_map m USING (patient_id)
JOIN stg.visit_map v USING (encounter_id);

CREATE OR REPLACE TABLE cdm.condition_occurrence AS
SELECT ROW_NUMBER() OVER (ORDER BY m.person_id, c.start_date, c.snomed_code) AS condition_occurrence_id,
       m.person_id, 0 AS condition_concept_id, c.start_date AS condition_start_date,
       c.stop_date AS condition_end_date, 32827 AS condition_type_concept_id, v.visit_occurrence_id,
       c.snomed_code AS condition_source_value, 0 AS condition_source_concept_id
FROM silver.conditions c
JOIN stg.person_map m USING (patient_id)
LEFT JOIN stg.visit_map v USING (encounter_id);

CREATE OR REPLACE TABLE cdm.measurement AS
SELECT ROW_NUMBER() OVER (ORDER BY m.person_id, o.obs_ts, o.loinc_code) AS measurement_id,
       m.person_id, 0 AS measurement_concept_id, o.obs_ts::DATE AS measurement_date,
       o.obs_ts AS measurement_datetime, 32827 AS measurement_type_concept_id,
       o.value_num AS value_as_number, 0 AS unit_concept_id, o.unit AS unit_source_value,
       v.visit_occurrence_id, o.loinc_code AS measurement_source_value, 0 AS measurement_source_concept_id
FROM silver.observations o
JOIN stg.person_map m USING (patient_id)
LEFT JOIN stg.visit_map v USING (encounter_id)
WHERE o.category IN ('vital-signs', 'laboratory')
  AND o.value_num IS NOT NULL
  AND regexp_full_match(o.loinc_code, '[0-9]+-[0-9]');

### Paso 6 — Módulos `transform.py` y `checks.py`

**Explicación línea por línea (`transform.py`)**

- `from importlib.resources import files`: lee archivos empaquetados (`sql/*.sql`) funcione donde funcione instalado el paquete.
- `def run_sql_file(con, name, **params):` carga el SQL, sustituye marcadores con `Template(...).substitute(**params)` (falla si falta un marcador: mejor un error que un SQL a medias) y ejecuta.
- `def export_schema(con, schema, tables, out_dir):` exporta tablas a Parquet con `COPY` y devuelve sus conteos.

**Explicación línea por línea (`checks.py`)**

- `class DataTestError(Exception)`: excepción propia; quien llame al pipeline puede distinguir "falló una prueba de datos" de "se cayó la red".
- `SILVER_TESTS` / `OMOP_TESTS`: diccionarios nombre → SQL que cuenta violaciones (convención de la semana 3).
- `def run_tests(con, tests):` ejecuta todas, registra cada resultado y, si alguna falla, lanza `DataTestError` con la lista completa (no solo la primera).

In [ ]:
%%writefile src/healthetl/transform.py
"""Transformaciones SQL (silver y OMOP) ejecutadas en DuckDB."""
import logging
from importlib.resources import files
from pathlib import Path
from string import Template

log = logging.getLogger(__name__)


def run_sql_file(con, name: str, **params) -> None:
    sql = files("healthetl").joinpath("sql").joinpath(name).read_text(encoding="utf-8")
    con.execute(Template(sql).substitute(**params))
    log.info("Ejecutado %s", name)


def export_schema(con, schema: str, tables: list, out_dir: Path) -> dict:
    out_dir.mkdir(parents=True, exist_ok=True)
    counts = {}
    for t in tables:
        dest = (out_dir / f"{t}.parquet").as_posix()
        con.execute(f"COPY {schema}.{t} TO '{dest}' (FORMAT parquet, COMPRESSION zstd)")
        counts[f"{schema}.{t}"] = con.execute(f"SELECT COUNT(*) FROM {schema}.{t}").fetchall()[0][0]
        log.info("%s.%s: %s filas", schema, t, f"{counts[f'{schema}.{t}']:,}")
    return counts

In [ ]:
%%writefile src/healthetl/checks.py
"""Pruebas de datos: cada SQL cuenta filas que violan una regla (0 = pasa)."""
import logging

log = logging.getLogger(__name__)


class DataTestError(Exception):
    """Una o más pruebas de datos fallaron."""


SILVER_TESTS = {
    "silver_patient_id_not_null": "SELECT COUNT(*) FROM silver.patients WHERE patient_id IS NULL",
    "silver_encounter_orphans": """SELECT COUNT(*) FROM silver.encounters e
        LEFT JOIN silver.patients p USING (patient_id) WHERE p.patient_id IS NULL""",
    "silver_start_parsed": """SELECT COUNT(*) FROM bronze.encounters
        WHERE START <> '' AND TRY_CAST(START AS TIMESTAMPTZ) IS NULL""",
    "silver_patients_reconciled": """SELECT ABS((SELECT COUNT(*) FROM bronze.patients)
        - (SELECT COUNT(*) FROM silver.patients))""",
}
OMOP_TESTS = {
    "omop_person_has_period": """SELECT COUNT(*) FROM cdm.person p
        LEFT JOIN cdm.observation_period op USING (person_id) WHERE op.person_id IS NULL""",
    "omop_visit_person_fk": """SELECT COUNT(*) FROM cdm.visit_occurrence v
        LEFT JOIN cdm.person p USING (person_id) WHERE p.person_id IS NULL""",
    "omop_gender_domain": "SELECT COUNT(*) FROM cdm.person WHERE gender_concept_id NOT IN (8507, 8532, 0)",
}


def run_tests(con, tests: dict) -> dict:
    results, failed = {}, []
    for name, sql in tests.items():
        n = con.execute(sql).fetchall()[0][0]
        results[name] = n
        log.info("%-32s %s (%s)", name, "PASS" if n == 0 else "FAIL", n)
        if n:
            failed.append(name)
    if failed:
        raise DataTestError(f"Pruebas fallidas: {failed}")
    return results

### Paso 7 — Orquestador `pipeline.py` con manifiesto de ejecución

**Explicación línea por línea**

- `def run(data_dir: Path) -> dict:` ejecuta todo en orden: extract → bronze → silver → pruebas silver → OMOP → pruebas OMOP → manifiesto.
- `run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")`: identificador único de ejecución (fecha-hora UTC).
- `t0 = time.perf_counter()`: reloj de alta precisión para medir duración.
- `con = duckdb.connect()` + `SET TimeZone = 'UTC'`: igual que en los notebooks.
- `transform.run_sql_file(con, "silver.sql", bronze_dir=bronze.as_posix())`: aplica el SQL de la semana 3.
- `checks.run_tests(con, checks.SILVER_TESTS)`: si falla, la excepción detiene el pipeline **antes** de publicar OMOP.
- `outputs = {p.name: extract.sha256_of(p) for p in ...}`: huella de cada Parquet publicado. Junto con la huella del ZIP de entrada, el manifiesto permite demostrar qué entrada produjo qué salida (**linaje** de punta a punta).
- `(data_dir / "runs" / f"{run_id}.json").write_text(...)`: guarda el manifiesto.
- `def main():` punto de entrada de línea de comandos con `argparse`: `python -m healthetl.pipeline --data-dir data`.
- `logging.basicConfig(level=logging.INFO, format=...)`: formato con hora, nivel y módulo.
- `if __name__ == "__main__": main()`: solo se ejecuta al llamar el archivo como programa, no al importarlo.

In [ ]:
%%writefile src/healthetl/pipeline.py
"""Orquestador: Synthea CSV -> bronze -> silver -> OMOP CDM v5.4, con pruebas y manifiesto."""
import argparse
import json
import logging
import platform
import time
from datetime import datetime, timezone
from pathlib import Path

import duckdb

from healthetl import checks, extract, transform

log = logging.getLogger("healthetl.pipeline")

SILVER_TABLES = ["patients", "encounters", "conditions", "observations"]
OMOP_TABLES = ["person", "observation_period", "visit_occurrence", "condition_occurrence", "measurement", "death"]


def run(data_dir: Path) -> dict:
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    t0 = time.perf_counter()
    raw_zip = extract.download(extract.DATASET_URL, data_dir / "raw" / "synthea_sample_data_csv_nov2021.zip")
    bronze = data_dir / "bronze"
    bronze_manifest = extract.extract_to_bronze(raw_zip, bronze, ingested_at=run_id)

    con = duckdb.connect()
    con.execute("SET TimeZone = 'UTC'")
    transform.run_sql_file(con, "silver.sql", bronze_dir=bronze.as_posix())
    silver_tests = checks.run_tests(con, checks.SILVER_TESTS)
    counts = transform.export_schema(con, "silver", SILVER_TABLES, data_dir / "silver")

    transform.run_sql_file(con, "omop.sql")
    omop_tests = checks.run_tests(con, checks.OMOP_TESTS)
    counts.update(transform.export_schema(con, "cdm", OMOP_TABLES, data_dir / "omop"))
    con.close()

    outputs = {p.name: extract.sha256_of(p) for p in sorted((data_dir / "omop").glob("*.parquet"))}
    manifest = {
        "run_id": run_id,
        "duration_s": round(time.perf_counter() - t0, 1),
        "python": platform.python_version(),
        "duckdb": duckdb.__version__,
        "input": {"url": extract.DATASET_URL, "sha256": extract.sha256_of(raw_zip)},
        "bronze": bronze_manifest,
        "tests": {**silver_tests, **omop_tests},
        "row_counts": counts,
        "omop_sha256": outputs,
    }
    runs = data_dir / "runs"
    runs.mkdir(parents=True, exist_ok=True)
    (runs / f"{run_id}.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    log.info("Ejecución %s completada en %ss", run_id, manifest["duration_s"])
    return manifest


def main():
    parser = argparse.ArgumentParser(description="Pipeline ETL healthetl")
    parser.add_argument("--data-dir", type=Path, default=Path("data"))
    args = parser.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
    run(args.data_dir.resolve())


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/healthetl/__init__.py
"""healthetl: pipeline ETL educativo de datos de salud (Synthea → OMOP CDM v5.4)."""
__version__ = "0.1.0"

### Paso 8 — Pruebas automáticas con `pytest`

**Explicación línea por línea (`tests/test_terminology.py`)**

- Cada función que empieza por `test_` es una prueba; `pytest` las encuentra solo.
- `assert verhoeff_ok("100005")`: ejemplos oficiales de la especificación SNOMED CT; si alguien "optimiza" la tabla y la rompe, la prueba lo detecta (es exactamente el error que se cometió y corrigió al preparar la semana 5).
- `@pytest.mark.parametrize("code", [...])`: ejecuta la misma prueba con varios valores; cada uno aparece como prueba independiente en el reporte.
- `assert loinc_check_digit("12345") == 5`: el ejemplo del manual de LOINC.

**Explicación línea por línea (`tests/test_checks.py`)**

- `con = duckdb.connect()` y `CREATE TABLE ... AS SELECT * FROM (VALUES ...)`: una base mínima en memoria con datos **construidos a mano**; la prueba no necesita descargar nada y tarda milisegundos.
- `test_detecta_huerfanos`: siembra un encuentro sin paciente y comprueba con `pytest.raises(DataTestError)` que la regla falla. Es la versión automatizada de la inyección de errores de la semana 2.
- `test_pasa_con_datos_limpios`: el caso contrario.

In [ ]:
%%writefile tests/test_terminology.py
import pytest

from healthetl.terminology import loinc_check_digit, loinc_ok, verhoeff_ok


@pytest.mark.parametrize("code", ["100005", "1290023401004", "11000001102", "999999990989121104", "59621000"])
def test_verhoeff_valid(code):
    assert verhoeff_ok(code)


@pytest.mark.parametrize("code", ["100006", "59621001", "abc", "12345"])
def test_verhoeff_invalid(code):
    assert not verhoeff_ok(code)


def test_loinc_manual_example():
    assert loinc_check_digit("12345") == 5


@pytest.mark.parametrize("code,expected", [("8480-6", True), ("8480-5", False), ("QALY", False), ("99999-1", False)])
def test_loinc_ok(code, expected):
    assert loinc_ok(code) is expected

In [ ]:
%%writefile tests/test_checks.py
import duckdb
import pytest

from healthetl.checks import DataTestError, run_tests

TESTS = {"orphans": """SELECT COUNT(*) FROM silver.encounters e
    LEFT JOIN silver.patients p USING (patient_id) WHERE p.patient_id IS NULL"""}


def make_db(encounter_patients):
    con = duckdb.connect()
    con.execute("CREATE SCHEMA silver")
    con.execute("CREATE TABLE silver.patients AS SELECT * FROM (VALUES ('p1'), ('p2')) t(patient_id)")
    values = ", ".join(f"('e{i}', '{p}')" for i, p in enumerate(encounter_patients))
    con.execute(f"CREATE TABLE silver.encounters AS SELECT * FROM (VALUES {values}) t(encounter_id, patient_id)")
    return con


def test_detecta_huerfanos():
    con = make_db(["p1", "p2", "p999"])
    with pytest.raises(DataTestError):
        run_tests(con, TESTS)


def test_pasa_con_datos_limpios():
    con = make_db(["p1", "p2"])
    assert run_tests(con, TESTS) == {"orphans": 0}

### Paso 9 — Instalar el paquete y ejecutar las pruebas

**Explicación línea por línea**

- `%pip install -q -e .`: instala el paquete en modo **editable** (`-e`): Python usa directamente tus archivos de `src/`, así cualquier cambio se refleja sin reinstalar. El `.` indica "el proyecto de esta carpeta" (lee `pyproject.toml`).
- `%pip install -q "pytest>=8"`: el ejecutor de pruebas.
- `!{sys.executable} -m pytest -q`: `!` ejecuta un comando del sistema; `{sys.executable}` es la ruta exacta del Python que corre este notebook (Jupyter sustituye la expresión entre llaves), así el comando usa el mismo intérprete donde instalaste el paquete y no otro Python del sistema; `-m pytest` ejecuta pytest como módulo; `-q` da un resumen corto. Debes ver todas las pruebas en verde (`passed`).

In [ ]:
%pip install -q -e .
%pip install -q "pytest>=8"

In [ ]:
!{sys.executable} -m pytest -q

### Paso 10 — Ejecutar el pipeline completo con un solo comando

**Explicación línea por línea**

- `!{sys.executable} -m healthetl.pipeline --data-dir "{DATA_DIR}"`: ejecuta el módulo como programa con el mismo intérprete del notebook; `{DATA_DIR}` lo sustituye Jupyter por la ruta de tu carpeta de datos. Fuera del notebook, el mismo comando es `python -m healthetl.pipeline --data-dir data`. Verás el registro con hora y nivel de cada paso. Como el ZIP ya existe, no se vuelve a descargar.
- Si una prueba de datos fallara, el comando terminaría con error y **no** se publicaría OMOP.

In [ ]:
!{sys.executable} -m healthetl.pipeline --data-dir "{DATA_DIR}"

### Paso 11 — Prueba de regresión: ¿el pipeline reproduce los notebooks?

**Explicación línea por línea**

- `runs = sorted((DATA_DIR / "runs").glob("*.json"))`: manifiestos de ejecución ordenados por nombre (= por fecha, gracias al formato del `run_id`).
- `last = json.loads(runs[-1].read_text())`: el más reciente.
- `expected = {...}`: conteos obtenidos en las semanas 3 y 6 con estos mismos datos.
- `for k, v in expected.items(): assert last["row_counts"][k] == v`: si alguien cambia el SQL y altera un conteo sin querer, esta prueba lo detecta. **En la práctica real** esto se llama prueba de regresión de datos.
- `print(json.dumps({...}, indent=2))`: muestra duración, versiones y huellas del manifiesto.

In [ ]:
import json

runs = sorted((DATA_DIR / "runs").glob("*.json"))
last = json.loads(runs[-1].read_text(encoding="utf-8"))
expected = {
    "silver.patients": 1163, "silver.encounters": 61459, "silver.observations": 530895,
    "cdm.person": 1163, "cdm.visit_occurrence": 61459, "cdm.measurement": 263767, "cdm.death": 163,
}
for k, v in expected.items():
    assert last["row_counts"][k] == v, f"{k}: {last['row_counts'][k]} != {v}"
print("Regresión OK: el pipeline reproduce los conteos de las semanas 3 y 6")
print(json.dumps({k: last[k] for k in ["run_id", "duration_s", "python", "duckdb", "input"]}, indent=2))

### Paso 12 — Integración continua con GitHub Actions

GitHub Actions ejecuta tus pruebas en un servidor de GitHub cada vez que haces `push`. Para repositorios públicos es gratuito.

**Explicación línea por línea (YAML)**

- `name: tests`: nombre del flujo.
- `on: [push, pull_request]`: se dispara en cada `push` y en cada *pull request*.
- `runs-on: ubuntu-latest`: máquina virtual Linux.
- `actions/checkout@v4`: descarga tu código. `actions/setup-python@v5` con `python-version: "3.12"`: instala Python.
- `pip install -e . pytest`: instala tu paquete y pytest.
- `python -m pytest -q`: ejecuta las pruebas. Si alguna falla, GitHub marca el *commit* en rojo.
- Las pruebas de `tests/` no descargan datos: por eso pueden correr en CI en segundos. El pipeline completo se ejecuta en Colab, no en CI.

In [ ]:
%%writefile .github/workflows/tests.yml
name: tests
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - run: pip install -e . pytest
      - run: python -m pytest -q

### Paso 13 — Publicar en GitHub (manual, fuera de este notebook)

Los comandos se muestran pero **no** se ejecutan aquí, porque requieren tu cuenta. Ejecútalos en una terminal (o en una celda con `!`) dentro de la carpeta del proyecto.

```bash
git init
git add .gitignore pyproject.toml requirements.txt src tests notebooks .github README.md
git status            # verifica que NO aparezca data/ ni ningún secreto
git commit -m "feat: pipeline healthetl Synthea → OMOP con pruebas y CI"
git branch -M main
git remote add origin https://github.com/<tu-usuario>/health-data-etl-lab.git
git push -u origin main
```

**Explicación línea por línea**

- `git init`: crea el repositorio local.
- `git add ...`: añade solo lo que quieres versionar; nunca `git add data`.
- `git status`: revisión obligatoria antes de cada *commit*.
- `git commit -m ...`: guarda una versión con un mensaje en formato *Conventional Commits* (`feat:`, `fix:`, `docs:`).
- `git branch -M main`, `git remote add origin ...`, `git push -u origin main`: nombra la rama, enlaza con GitHub y sube. GitHub pide un *Personal Access Token* en lugar de contraseña; guárdalo en Colab Secrets, nunca en el notebook.
- Para los notebooks, Colab ofrece además **Archivo → Guardar una copia en GitHub**.

### Autoevaluación final

1. ¿Qué ventajas tiene `logging` sobre `print` en un pipeline nocturno?
2. ¿Qué te permite demostrar el manifiesto de ejecución ante una auditoría?
3. ¿Por qué las pruebas de CI usan datos construidos a mano y no Synthea?
4. Nombra dos herramientas de orquestación de la industria que reemplazarían a `pipeline.py` a mayor escala y qué te darían.

**Commit sugerido:** `feat(s08): paquete healthetl con pruebas pytest, manifiesto de ejecución y CI`